In [ ]:
# PHẦN 1: SETUP MÔI TRƯỜNG & TẢI DỮ LIỆU KAGGLE

import os
import shutil
import json
import torch
import gc
import time
import nltk
from google.colab import drive
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback
)
from transformers.trainer_utils import get_last_checkpoint
from datasets import Dataset

os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY'] = "0ba946628cb1f5acb76ecd357f590e95"

drive.mount('/content/drive')


FINAL_SAVE_PATH = "/content/drive/My Drive/T5_Large_Spider_CRP_FFT"
CHECKPOINT_DIR = "/content/drive/My Drive/T5_Large_Spider_CRP_FFT/checkpoints"

print(">>> [1/5] Kiểm tra và tải dữ liệu...")
if not os.path.exists('spider_data'):
    !pip install -q kaggle
    !kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
    !unzip -q yale-universitys-spider-10-nlp-dataset.zip -d temp_spider
    if os.path.exists("temp_spider/spider"):
        shutil.move("temp_spider/spider", "spider_data")
        shutil.rmtree('temp_spider')
        !wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py
        !wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py
        nltk.download('punkt')
        nltk.download('punkt_tab')
    print("✅ Dữ liệu đã sẵn sàng.")
else:
    print("📂 Dữ liệu đã tồn tại, bỏ qua bước tải.")


# PHẦN 2: PREPROCESSING (CODE REPRESENTATION - CR_P)

MODEL_NAME = "t5-large"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

with open('spider_data/tables.json', 'r') as f:
    table_data = json.load(f)
schema_map = {db['db_id']: db for db in table_data}

def get_crp_schema(db_id):
    """Tạo cấu trúc CREATE TABLE đầy đủ (Types, PK, FK) theo chuẩn DAIL-SQL."""
    if db_id not in schema_map: return ""
    db = schema_map[db_id]
    ddl_statements = []
    for i, table_name in enumerate(db['table_names_original']):
        col_defs = [f"{c[1]} {db['column_types'][j]}" for j, c in enumerate(db['column_names_original']) if c[0] == i]
        pk_idx = db['primary_keys']
        pks = [db['column_names_original'][j][1] for j in pk_idx if db['column_names_original'][j][0] == i]
        if pks: col_defs.append(f"primary key ({', '.join(pks)})")
        for fk in db['foreign_keys']:
            if db['column_names_original'][fk[0]][0] == i:
                from_col = db['column_names_original'][fk[0]][1]
                to_table = db['table_names_original'][db['column_names_original'][fk[1]][0]]
                to_col = db['column_names_original'][fk[1]][1]
                col_defs.append(f"foreign key ({from_col}) references {to_table}({to_col})")
        ddl_statements.append(f"CREATE TABLE {table_name} ({', '.join(col_defs)});")
    return " ".join(ddl_statements)

def load_spider_unified(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return Dataset.from_dict({
        "question": [item["question"] for item in data],
        "query": [item["query"] for item in data],
        "db_id": [item["db_id"] for item in data]
    })

def preprocess_fn(examples):
    inputs = [f"translate to SQL: {q} | Schema: {get_crp_schema(d)}" for q, d in zip(examples['question'], examples['db_id'])]
    model_inputs = tokenizer(inputs, max_length=512, padding="max_length", truncation=True)
    labels = tokenizer(examples['query'], max_length=128, padding="max_length", truncation=True)
    labels["input_ids"] = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = load_spider_unified("spider_data/train_spider.json").map(preprocess_fn, batched=True)
val_ds = load_spider_unified("spider_data/dev.json").map(preprocess_fn, batched=True)

# PHẦN 3: TRAINING FFT (HỖ TRỢ RESUME TỪ CHECKPOINT)

print(f"\n>>> [3/5] Khởi tạo huấn luyện {MODEL_NAME}...")

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    optim="adafactor",
    weight_decay=0.01,
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    report_to="none",
    logging_steps=10
)

trainer = Seq2SeqTrainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds)

# LOGIC RESUME CHECKPOINT

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if last_checkpoint is not None:
    print(f"🔄 Tìm thấy checkpoint: {last_checkpoint}. Đang tiếp tục...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🚀 Không có checkpoint cũ. Bắt đầu train mới...")
    trainer.train()

trainer.save_model(FINAL_SAVE_PATH)
tokenizer.save_pretrained(FINAL_SAVE_PATH)


# PHẦN 4: INFERENCE (DỰ ĐOÁN)

print("\n>>> [4/5] Chạy Inference trên tập Validation...")
model.eval()
predictions, gold_lines = [], []

for item in val_ds:
    input_text = f"translate to SQL: {item['question']} | Schema: {get_crp_schema(item['db_id'])}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    predictions.append(tokenizer.decode(output[0], skip_special_tokens=True) + "\n")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

# PHẦN 5: CHẤM ĐIỂM (EVALUATION)

print("\n>>> [5/5] Kết quả đánh giá cuối cùng:")
!sed -i 's/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")/' evaluation.py
!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all

Mounted at /content/drive
>>> [1/5] Kiểm tra và tải dữ liệu...
Dataset URL: https://www.kaggle.com/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset
License(s): unknown
  0% 0.00/96.0M [00:00<?, ?B/s]
100% 96.0M/96.0M [00:00<00:00, 1.60GB/s]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ Dữ liệu đã sẵn sàng.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
15,0.208126,0.535877


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


>>> [4/5] Chạy Inference trên tập Validation...

>>> [5/5] Kết quả đánh giá cuối cùng:
eval_err_num:1
medium pred: SELECT name , capacity FROM stadium ORDER BY avg(attendance) DESC LIMIT 1
medium gold: SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1

eval_err_num:2
medium pred: SELECT name , capacity FROM stadium ORDER BY avg(attendance) DESC LIMIT 1
medium gold: SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1

extra pred: SELECT T2.name , T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id = T2.stadium_id WHERE T1.year >= 2014 GROUP BY T2.name ORDER BY count(*) DESC LIMIT 1
extra gold: SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.year  >=  2014 GROUP BY T2.stadium_id ORDER BY count(*) DESC LIMIT 1

extra pred: SELECT T2.name , T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id = T2.stadium_id WHERE T1.year > 2013 GROUP BY T1.stadium_id ORDER BY count

In [2]:
# ĐÁNH GIÁ HIỆU NĂNG CHO T5-LARGE

import torch
import time
import numpy as np
import psutil
import os
import gc
from transformers import T5Tokenizer, T5ForConditionalGeneration

from google.colab import drive
drive.mount('/content/drive')
print("Đang dọn dẹp bộ nhớ GPU...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# 1. Đường dẫn của T5-Large
FINAL_SAVE_PATH = "/content/drive/My Drive/T5_Large_Spider_CRP_FFT"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Đang tải mô hình từ: {FINAL_SAVE_PATH}")

# 2. Tải mô hình
try:
    tokenizer = T5Tokenizer.from_pretrained(FINAL_SAVE_PATH)
    model = T5ForConditionalGeneration.from_pretrained(FINAL_SAVE_PATH)
    model.to(device)
    model.eval()
    print(" Tải mô hình T5-Large thành công!")
except Exception as e:
    print(f" Lỗi: {e}")
    exit()

# 3. Dữ liệu đầu vào
sample_question = "How many students are there?"
sample_schema = "CREATE TABLE student(id int, name text);"
input_text = f"translate to SQL: {sample_question} | Schema: {sample_schema}"

inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)

# 4. Warmup GPU
print("Đang warmup GPU...")
for _ in range(10):
    with torch.no_grad():
        model.generate(**inputs, max_length=128)

# 5. Đo Latency
print("Đang đo lường độ trễ (100 lần chạy)...")
latencies = []
for _ in range(100):
    start = time.time()
    with torch.no_grad():
        model.generate(**inputs, max_length=128)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latencies.append(time.time() - start)

avg_latency = np.mean(latencies)
std_latency = np.std(latencies)

# 6. Thông lượng (Throughput)
throughput = 1 / avg_latency

# 7. Đo VRAM và RAM
gpu_memory_allocated = None
if torch.cuda.is_available():
    gpu_memory_allocated = torch.cuda.max_memory_allocated() / 1024**2

process = psutil.Process(os.getpid())
ram_usage = process.memory_info().rss / 1024**2
total_params = sum(p.numel() for p in model.parameters())

# 8. In kết quả
print("\n" + "="*45)
print("     ĐÁNH GIÁ HIỆU NĂNG (T5-LARGE)      ")
print("="*45)
print(f"Độ trễ TB (Latency):            {avg_latency*1000:.2f} ms")
print(f"Độ lệch chuẩn (Std):            {std_latency*1000:.2f} ms")
print(f"Thông lượng (Throughput BS=1):  {throughput:.2f} samples/sec")
if gpu_memory_allocated:
    print(f"VRAM tối đa đã dùng (Peak):     {gpu_memory_allocated:.2f} MB")
print(f"RAM CPU đang sử dụng:           {ram_usage:.2f} MB")
print(f"Tổng số tham số mô hình:        {total_params:,}")
print("="*45)

Mounted at /content/drive
Đang dọn dẹp bộ nhớ GPU...
Đang tải mô hình từ: /content/drive/My Drive/T5_Large_Spider_CRP_FFT


Loading weights:   0%|          | 0/509 [00:02<?, ?it/s]

✅ Tải mô hình T5-Large thành công!
Đang warmup GPU...
Đang đo lường độ trễ (100 lần chạy)...

     ĐÁNH GIÁ HIỆU NĂNG (T5-LARGE)      
Độ trễ TB (Latency):            432.72 ms
Độ lệch chuẩn (Std):            309.07 ms
Thông lượng (Throughput BS=1):  2.31 samples/sec
VRAM tối đa đã dùng (Peak):     2830.79 MB
RAM CPU đang sử dụng:           1540.35 MB
Tổng số tham số mô hình:        737,668,096
